In [ ]:
!pip install --quiet datasets transformers peft ipython numpy matplotlib evaluate jiwer librosa tensorboard

In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
from scipy.signal import resample
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader
from transformers import WhisperTokenizer, WhisperFeatureExtractor, WhisperForConditionalGeneration
import evaluate
import pandas as pd
from datasets import Dataset, Audio
import os
from peft import LoraConfig, get_peft_model, PeftModel # Added PeftModel import
import librosa
from torch.utils.tensorboard import SummaryWriter
import datetime
import json

In [ ]:
# --- Configuration for Resumption ---
RESUME_FROM_CHECKPOINT = True
KAGGLE_PREV_OUTPUT_PATH = "/kaggle/input/checkpoints-3"
CHECKPOINT_PATH = os.path.join(KAGGLE_PREV_OUTPUT_PATH, "best_lora_model") 
OPTIMIZER_STATE_PATH = os.path.join(CHECKPOINT_PATH, "optimizer_state.pt")

START_EPOCH = 25

KAGGLE_WORKING_DIR = "/kaggle/working"
os.makedirs(os.path.join(KAGGLE_WORKING_DIR, "losses"), exist_ok=True)
os.makedirs(os.path.join(KAGGLE_WORKING_DIR, "lora_checkpoints"), exist_ok=True)

In [ ]:
def down_sample_audio(audio_original, original_sample_rate):
    target_sample_rate = 16000
    num_samples = int(len(audio_original) * target_sample_rate / original_sample_rate)
    downsampled_audio = resample(audio_original, num_samples)
    return downsampled_audio

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language='ar', task='transcribe')
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small", language='ar', task='transcribe')

In [ ]:
base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False
base_model.to('cuda')

print('Base model initialized.')

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    inference_mode=False,
)
print('LoRA config done.')

model = None
print(f"RESUME_FROM_CHECKPOINT: {RESUME_FROM_CHECKPOINT}")
print(f"Checkpoint path exists: {os.path.exists(CHECKPOINT_PATH)}")

# Print checkpoint contents for debugging
if os.path.exists(CHECKPOINT_PATH):
    print(f"Checkpoint directory contents: {os.listdir(CHECKPOINT_PATH)}")
    if os.path.exists(os.path.join(CHECKPOINT_PATH, "adapter_config.json")):
        with open(os.path.join(CHECKPOINT_PATH, "adapter_config.json"), "r") as f:
            adapter_config = json.load(f)
        print("Checkpoint LoRA config:", adapter_config)

In [ ]:
# Load model
if RESUME_FROM_CHECKPOINT and os.path.exists(CHECKPOINT_PATH):
    print(f"Attempting to resume training from checkpoint: {CHECKPOINT_PATH}")
    try:
        # Load PeftModel
        model = PeftModel.from_pretrained(base_model, CHECKPOINT_PATH)
        print("LoRA adapters loaded from checkpoint.")
        
        # Set requires_grad = True for LoRA parameters
        for name, param in model.named_parameters():
            if "lora" in name:
                param.requires_grad = True
        
        print("\n--- Trainable parameters AFTER loading checkpoint ---")
        model.print_trainable_parameters()  # Should show ~3.5M trainable params
        print("--------------------------------------------------\n")
        
        # Ensure model is in training mode
        model.train()
        
    except Exception as e:
        print(f"Error loading PeftModel: {e}")
        print("Initializing new LoRA model.")
        model = get_peft_model(base_model, lora_config)
        START_EPOCH = 0
        print("\n--- Trainable parameters AFTER FALLBACK to new LoRA model ---")
        model.print_trainable_parameters()
        print("----------------------------------------------------------\n")
else:
    print("RESUME_FROM_CHECKPOINT is False or checkpoint path not found. Initializing new LoRA model.")
    model = get_peft_model(base_model, lora_config)
    START_EPOCH = 0
    print("New LoRA model initialized.")
    print("\n--- Trainable parameters for NEW LoRA model ---")
    model.print_trainable_parameters()
    print("-------------------------------------------\n")

In [ ]:
# --- Initialize Optimizer ---
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

# Load optimizer state
if RESUME_FROM_CHECKPOINT and os.path.exists(OPTIMIZER_STATE_PATH):
    print(f"Loading optimizer state from: {OPTIMIZER_STATE_PATH}")
    try:
        optimizer_state = torch.load(OPTIMIZER_STATE_PATH)
        optimizer.load_state_dict(optimizer_state)
        print("Optimizer state loaded successfully.")
        print("Optimizer state keys:", optimizer_state.keys())
    except Exception as e:
        print(f"Error loading optimizer state: {e}. Starting with fresh optimizer.")
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
else:
    print("Optimizer state not loaded (starting new training or path not found).")
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

In [ ]:
# Data Loading
KAGGLE_INPUT_ROOT = "/kaggle/input"
CSV_PATH = os.path.join(KAGGLE_INPUT_ROOT, "linto-dataset", "linto_dataset", "linto_segmented_dataset.csv")
path_updated = os.path.join(KAGGLE_INPUT_ROOT, "linto-dataset", "linto_dataset")

def create_cleaned_audio_path(relative_path, base_dir):
    relative_path_normalized = relative_path.replace("\\", "/")
    full_path_with_original_ext = os.path.join(base_dir, relative_path_normalized)
    root, ext = os.path.splitext(full_path_with_original_ext)
    cleaned_full_path = root + ".wav"
    return os.path.normpath(cleaned_full_path)

print(f"Loading data from: {CSV_PATH}")
try:
    small_df = pd.read_csv(CSV_PATH)
except FileNotFoundError:
    print(f"Error: CSV file not found at {CSV_PATH}.")
    print("Please ensure your dataset is correctly linked and path is accurate.")
    exit()

small_df['audio_path'] = small_df['wav'].apply(lambda x: create_cleaned_audio_path(x, path_updated))

print(f"Example of a constructed audio path: {small_df.iloc[0]['audio_path']}")
print("\nVerifying audio file existence and filtering dataset...")
valid_rows = []
skipped_audio_count = 0

for index, row in tqdm(small_df.iterrows(), total=len(small_df), desc="Checking audio files"):
    audio_full_path = row['audio_path']
    audio_id = os.path.splitext(os.path.basename(audio_full_path))[0]
    if os.path.exists(audio_full_path):
        valid_rows.append(row)
    else:
        skipped_audio_count += 1

small_df2 = pd.DataFrame(valid_rows)
if skipped_audio_count > 0:
    print(f"\nTotal {skipped_audio_count} audio files were skipped due to not being found.")
    print(f"Proceeding with {len(small_df2)} valid entries in the DataFrame.")
else:
    print("\nAll constructed audio paths found. No files skipped during initial verification.")

In [ ]:
data_list_for_hf_dataset = []

for index, row in small_df2.iterrows():
    audio_id_from_path = os.path.splitext(os.path.basename(row['audio_path']))[0]
    data_list_for_hf_dataset.append({
        "audio": {"path": row['audio_path']},
        "sentence": row['wrd'],
        "audio_id": audio_id_from_path
    })

custom_hf_dataset = Dataset.from_list(data_list_for_hf_dataset)
custom_hf_dataset = custom_hf_dataset.cast_column("audio", Audio(sampling_rate=16000))

split_custom_dataset = custom_hf_dataset.train_test_split(test_size=0.2, seed=42)

train_data = split_custom_dataset['train']
test_data = split_custom_dataset['test']

print(f"Loaded {len(train_data)} training examples and {len(test_data)} test examples from your local dataset for testing.")

In [ ]:
list_of_transcription_lengths = []
tokenized_text = tokenizer(train_data['sentence']).input_ids
for text in tokenized_text:
    list_of_transcription_lengths.append(len(text))

plt.hist(list_of_transcription_lengths)
plt.xlabel("sentence length")
plt.ylabel("number of transcripts")
plt.title("Distribution of Tokenized Transcription Lengths")
plt.show()

In [ ]:
MAX_SEQ_LEN = 140

class whisper_training_dataset(torch.utils.data.Dataset):
    def __init__(self, dataset, max_len):
        self.dataset = dataset
        self.max_len = max_len
        self.bos_token = model.config.decoder_start_token_id

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        try:
            item = self.dataset[idx]
            
            if 'audio' not in item or item['audio'] is None or 'array' not in item['audio'] or 'sampling_rate' not in item['audio']:
                print(f"Skipping item {idx} due to incomplete audio data after loading (should not happen after pre-filtering).")
                return None
            
            audio_data = down_sample_audio(item['audio']["array"], item['audio']["sampling_rate"])
            
            features = feature_extractor(raw_speech=audio_data, sampling_rate=16000, return_tensors='pt')
            input_features = features.input_features[0]
            input_attention_mask = features.attention_mask[0] if 'attention_mask' in features and features.attention_mask is not None else None

            transcription = item["sentence"]

            labels = tokenizer(transcription, padding="max_length", max_length=self.max_len, truncation=True, return_tensors="pt")
            labels = labels["input_ids"].masked_fill(labels['attention_mask'].ne(1), -100)
            labels = labels[0][1:]

            return {
                "input_features": input_features,
                "input_attention_mask": input_attention_mask,
                "labels": labels
            }
        except Exception as e:
            print(f"Skipping item {idx} due to an unexpected error during processing: {e}")
            return None

In [ ]:
def collate_fn(batch):
    batch = [item for item in batch if item is not None]
    if not batch:
        return {}

    input_features = torch.stack([x['input_features'] for x in batch])
    labels = torch.stack([x['labels'] for x in batch])
    
    if batch[0]["input_attention_mask"] is not None:
        input_attention_mask = torch.stack([x['input_attention_mask'] for x in batch])
    else:
        input_attention_mask = None
    
    return {"input_features": input_features, "input_attention_mask": input_attention_mask, "labels": labels}

train_whisper_dataset = whisper_training_dataset(dataset=train_data, max_len=MAX_SEQ_LEN)

train_dataloader = torch.utils.data.DataLoader(
    train_whisper_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

In [ ]:
from jiwer import wer

def evaluation(model):
    device='cuda'
    
    test_dataset = whisper_training_dataset(dataset=test_data, max_len=MAX_SEQ_LEN)
    test_dataloader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=8,
        shuffle=False,
        collate_fn=collate_fn,
    )
    
    model.eval()
    predictions=[]
    references=[]

    for batch in tqdm(test_dataloader,total=len(test_dataloader)):
        input_features = batch["input_features"].to(device)
        labels = batch["labels"].to(device)
        
        input_attention_mask = batch["input_attention_mask"]
        if input_attention_mask is not None:
            input_attention_mask = input_attention_mask.to(device)

        with torch.no_grad():
            generated_tokens = model.generate(
                input_features=input_features,
                attention_mask=input_attention_mask,
                language='arabic',
                task='transcribe',
                max_new_tokens=MAX_SEQ_LEN
            )

        decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        labels = labels.cpu().numpy()
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        predictions.extend(decoded_preds)
        references.extend(decoded_labels)
        
    WER = wer(references, predictions) * 100
    return WER

In [ ]:
# TensorBoard setup
total_batches_per_epoch = len(train_dataloader)
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
TENSORBOARD_LOG_DIR = os.path.join(KAGGLE_WORKING_DIR, 'runs', current_time) 
os.makedirs(TENSORBOARD_LOG_DIR, exist_ok=True)
writer = SummaryWriter(log_dir=TENSORBOARD_LOG_DIR)
print(f"TensorBoard logs will be saved to: {TENSORBOARD_LOG_DIR}")

torch.cuda.empty_cache()
model.train()
device='cuda'

# --- Training Loop ---
max_epochs = 30
best_wer = float('inf')
patience = 5  
patience_counter = 0
epoch_metrics = []
running_wer_per_epoch = []

for epoch in range(START_EPOCH, max_epochs):
    print(f"\nEpoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_train_losses = []
    for batch_idx, batch in enumerate(tqdm(train_dataloader, total=total_batches_per_epoch, leave=False, desc=f"Epoch {epoch + 1}")):
        if not batch:
            continue
        input_features, labels = batch["input_features"].to(device), batch["labels"].to(device)
        outputs = model(input_features, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        global_step = epoch * total_batches_per_epoch + batch_idx
        writer.add_scalar('Loss/Train_Step_Loss', loss.item(), global_step)
        epoch_train_losses.append(loss.item())
        
        if (batch_idx + 1) % 2500 == 0:
            print(f"Epoch {epoch + 1}, Step {batch_idx + 1}/{len(train_dataloader)}, Loss: {loss.item():.4f}")
            save_path = os.path.join(KAGGLE_WORKING_DIR, f'lora_checkpoints/lora_model_epoch_{epoch+1}_step_{batch_idx+1}')
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            model.save_pretrained(save_path, safe_serialization=True)
            # Convert adapter_config to dict and ensure target_modules is a list
            adapter_config = lora_config.to_dict()
            adapter_config["inference_mode"] = False
            adapter_config["target_modules"] = list(adapter_config["target_modules"])  # Convert set to list
            with open(os.path.join(save_path, "adapter_config.json"), "w") as f:
                json.dump(adapter_config, f, indent=2)
            print(f"LoRA adapters saved to '{save_path}'")
            
    torch.cuda.empty_cache()
    current_wer = evaluation(model)
    running_wer_per_epoch.append(current_wer)
    print(f"End of Epoch {epoch + 1}, Validation WER: {current_wer:.2f}%")
    writer.add_scalar('WER/Validation_Epoch_WER', current_wer, epoch)
    avg_epoch_train_loss = np.mean(epoch_train_losses) if epoch_train_losses else 0
    writer.add_scalar('Loss/Train_Epoch_Loss', avg_epoch_train_loss, epoch)
    epoch_metrics.append({
        'epoch': epoch + 1,
        'avg_training_loss': avg_epoch_train_loss,
        'validation_wer': current_wer
    })
    
    if current_wer < best_wer:
        best_wer = current_wer
        patience_counter = 0
        best_model_save_path = os.path.join(KAGGLE_WORKING_DIR, 'best_lora_model')
        os.makedirs(best_model_save_path, exist_ok=True)
        model.save_pretrained(best_model_save_path, safe_serialization=True)
        # Convert adapter_config to dict and ensure target_modules is a list
        adapter_config = lora_config.to_dict()
        adapter_config["inference_mode"] = False
        adapter_config["target_modules"] = list(adapter_config["target_modules"])  # Convert set to list
        with open(os.path.join(best_model_save_path, "adapter_config.json"), "w") as f:
            json.dump(adapter_config, f, indent=2)
        torch.save(optimizer.state_dict(), os.path.join(best_model_save_path, "optimizer_state.pt"))
        print(f"New best model and optimizer state saved to '{best_model_save_path}' with WER: {best_wer:.2f}")
    else:
        patience_counter += 1
        print(f"Validation WER did not improve. Patience: {patience_counter}/{patience}")
    if patience_counter >= patience:
        print(f"Early stopping triggered! No improvement for {patience} consecutive epochs.")
        break

writer.close()
print("\nLoRA Fine-tuning complete!")

In [ ]:
# --- Save epoch metrics to CSV ---
# This CSV will only contain metrics from the resumed run (Epoch 7 onwards)
metrics_df = pd.DataFrame(epoch_metrics)
metrics_csv_path = os.path.join(KAGGLE_WORKING_DIR, 'training_metrics_resumed.csv') # Use a new name for clarity
metrics_df.to_csv(metrics_csv_path, index=False)
print(f"Training metrics saved to: {metrics_csv_path}")

# --- Final Plots ---
# Plot Validation WER over Epochs (will only show epochs from START_EPOCH onwards)
plt.figure(figsize=(10, 5))
# Plotting range should be adjusted to reflect START_EPOCH
plt.plot(range(START_EPOCH + 1, START_EPOCH + 1 + len(running_wer_per_epoch)), running_wer_per_epoch, marker='o', linestyle='-')
plt.xlabel('Epoch')
plt.ylabel('Validation WER')
plt.title(f'Validation WER Over Epochs (Resumed from Epoch {START_EPOCH + 1})')
plt.grid(True)
plt.show()

print("\nWER values per epoch (from start of this resumed run):", running_wer_per_epoch)

# --- Instructions for TensorBoard Access ---
print("\nTo access TensorBoard:")
print("1. After the notebook run is complete, go to the 'Output' tab.")
print("2. In the 'Output Data' section on the right, navigate into the 'runs' folder, then into the latest timestamped folder (e.g., '20250731-HHMMSS').")
print("3. You should see a 'TensorBoard' button appear at the top of that folder view. Click it to launch TensorBoard.")
print(f"Your logs for this run are located at: {TENSORBOARD_LOG_DIR}")